# 76: Position Sizing Investigation

**Date:** 2026-01-22  
**Purpose:** Test gradual position scaling instead of binary all-in/all-out

## The Problem with Binary Signals:

Current approach:
- **Entry:** 4/5 conditions = 100% in, else 0%
- **Exit:** Signal triggers = 100% out instantly
- **Issue:** All-or-nothing timing risk

## Position Sizing Alternatives:

### 1. Signal Strength Sizing
- 3/5 conditions = 25% position
- 4/5 conditions = 50% position
- 5/5 conditions = 100% position

### 2. Gradual Scaling
- Scale in: Add 25% every 7 days if conditions persist
- Scale out: Remove 25% every 7 days if exit conditions meet

### 3. Hybrid Approach
- Keep 50% in permanent buy-and-hold
- Trade 50% using Check framework signals

### 4. Kelly Criterion
- Calculate optimal position size based on win rate and avg win/loss
- Win rate: 100%, Avg win: TBD → Kelly %

### 5. Confidence-Based Sizing
- More metrics triggered = higher conviction = larger position
- Entry: 20% per condition met (max 100%)
- Exit: 15% per metric triggered (7 metrics max)

## Expected Benefits:

- ✅ Reduce timing risk (dollar-cost average in/out)
- ✅ Stay partially invested (capture some upside)
- ✅ Smoother equity curve
- ✅ Lower psychological stress
- ⚠️ May reduce peak returns
- ⚠️ More complex to execute

In [ ]:
# Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# VectorBT (mandatory)
try:
    import vectorbt as vbt
    print("✓ VectorBT loaded")
except ImportError:
    print("✗ Installing VectorBT...")
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "vectorbt"])
    import vectorbt as vbt
    print("✓ VectorBT installed")

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

print("\n✓ Setup complete!")

In [ ]:
# Configuration
PROJECT_ROOT = Path().resolve().parent
DATA_DIR = PROJECT_ROOT / "data" / "brk" / "daily"
GLASSNODE_DIR = PROJECT_ROOT / "data" / "glassnode" / "daily"

# Test period (out-of-sample)
TEST_START = '2023-01-01'

# Costs
FEES = 0.001
SLIPPAGE = 0.001

print(f"Testing period: {TEST_START} onwards")
print(f"Fees: {FEES*100}%, Slippage: {SLIPPAGE*100}%")

## 1. Load Data

In [ ]:
def load_metric(name: str, source: str = "brk") -> pd.Series:
    """Load metric as time-indexed Series."""
    if source == "brk":
        path = DATA_DIR / f"{name}.parquet"
    elif source == "glassnode":
        path = GLASSNODE_DIR / f"{name}.parquet"
    else:
        return pd.Series(dtype=float)
    
    if not path.exists():
        return pd.Series(dtype=float)
    
    df = pd.read_parquet(path)
    
    if 'time' not in df.columns and isinstance(df.index, pd.DatetimeIndex):
        df = df.reset_index()
        if len(df.columns) == 2:
            df.columns = ['time', 'value']
    
    if 'value' not in df.columns:
        for col in df.columns:
            if col != 'time' and pd.api.types.is_numeric_dtype(df[col]):
                df['value'] = df[col]
                break
    
    if 'time' in df.columns and 'value' in df.columns:
        df['time'] = pd.to_datetime(df['time'])
        df = df.set_index('time')['value']
        df = df.sort_index()
        return df
    
    return pd.Series(dtype=float)


# Load metrics
print("Loading data...")

metrics = {
    'price': ('price', 'brk'),
    'mvrv': ('mvrv', 'brk'),
    'mvrv_sth': ('mvrv_sth', 'brk'),
    'mvrv_lth': ('mvrv_lth', 'brk'),
    'sopr': ('sopr', 'brk'),
    'sopr_sth': ('sopr_sth', 'brk'),
    'sopr_lth': ('sopr_lth', 'brk'),
    'realized_profit': ('realized_profit', 'brk'),
    'realized_loss': ('realized_loss', 'brk'),
    'puell': ('puell_multiple', 'brk'),
    'price_200sma': ('price_200d_sma', 'brk'),
    'funding': ('funding_rate', 'glassnode'),
    'liq_long': ('liquidations_long', 'glassnode'),
    'liq_short': ('liquidations_short', 'glassnode'),
}

df_dict = {}
for name, (metric, source) in metrics.items():
    series = load_metric(metric, source)
    if not series.empty:
        df_dict[name] = series
        print(f"  ✓ {name}")

df = pd.DataFrame(df_dict)
df = df.fillna(method='ffill')
df = df[df.index >= TEST_START].copy()

print(f"\n✓ Data: {len(df)} days ({df.index[0].date()} to {df.index[-1].date()})")
df.head()

## 2. Calculate Individual Condition Strength

In [ ]:
def calculate_rolling_zscore(series: pd.Series, window: int) -> pd.Series:
    """Calculate rolling z-score."""
    mean = series.rolling(window, min_periods=30).mean()
    std = series.rolling(window, min_periods=30).std()
    return (series - mean) / std


# Entry conditions (Buy The Dip)
entry_c1 = df['mvrv_sth'] < 1.0
entry_c2 = df['sopr_sth'] < 1.0
entry_c3 = (df['realized_profit'] / df['realized_loss']) < 1.0
entry_c4 = df['funding'] <= 0.0
entry_c5 = (df['liq_long'] / df['liq_short']) > 1.0

# Count conditions (0-5)
entry_count = (
    entry_c1.astype(int) + 
    entry_c2.astype(int) + 
    entry_c3.astype(int) + 
    entry_c4.astype(int) + 
    entry_c5.astype(int)
)

# Exit conditions (8-Metric Z-scores)
mvrv_z = calculate_rolling_zscore(df['mvrv'], 1460)
mvrv_sth_z = calculate_rolling_zscore(df['mvrv_sth'], 365)
sopr_z = calculate_rolling_zscore(df['sopr'], 365)
sopr_sth_z = calculate_rolling_zscore(df['sopr_sth'], 365)
puell_z = calculate_rolling_zscore(df['puell'], 365)
mayer_z = calculate_rolling_zscore(df['price'] / df['price_200sma'], 365)
funding_z = calculate_rolling_zscore(df['funding'], 365)

# Count exit triggers (0-7)
exit_count = (
    (mvrv_z > 1.5).astype(int) +
    (mvrv_sth_z > 1.25).astype(int) +
    (sopr_z > 1.5).astype(int) +
    (sopr_sth_z > 1.0).astype(int) +
    (mayer_z > 1.0).astype(int) +
    (puell_z > 1.5).astype(int) +
    (funding_z > 1.5).astype(int)
)

# LTH Distribution exit (binary)
lth_exit = (df['mvrv'] > 2.0) & (df['sopr_lth'] > 1.5)

print(f"Entry condition distribution:")
print(entry_count.value_counts().sort_index())
print(f"\nExit trigger distribution:")
print(exit_count.value_counts().sort_index())

## 3. Strategy 1: Signal Strength Sizing

Position size based on number of conditions met.

In [ ]:
def signal_strength_sizing(entry_count: pd.Series, exit_count: pd.Series) -> pd.Series:
    """
    Position size based on signal strength.
    
    Entry:
    - 0-2/5: 0% (no position)
    - 3/5: 40%
    - 4/5: 70%
    - 5/5: 100%
    
    Exit:
    - 0-3 metrics: Keep position
    - 4-5 metrics: Scale down to 50%
    - 6-7 metrics: Exit completely (0%)
    """
    position_size = pd.Series(0.0, index=entry_count.index)
    
    # Entry sizing
    position_size[entry_count == 3] = 0.40
    position_size[entry_count == 4] = 0.70
    position_size[entry_count == 5] = 1.00
    
    # Exit scaling
    position_size[(exit_count >= 4) & (exit_count <= 5)] = position_size.clip(upper=0.50)
    position_size[exit_count >= 6] = 0.0
    
    return position_size


# Generate position sizes
position_sizes = signal_strength_sizing(entry_count, exit_count)

# Run VectorBT backtest with variable position sizing
pf_signal_strength = vbt.Portfolio.from_orders(
    close=df['price'],
    size=position_sizes,
    size_type='targetpercent',  # Target % of portfolio
    fees=FEES,
    slippage=SLIPPAGE,
    init_cash=10000,
    freq='1D'
)

# Results
print("\nSignal Strength Sizing Results:")
print("="*60)
print(f"Total Return: {pf_signal_strength.total_return() * 100:.1f}%")
print(f"Sharpe Ratio: {pf_signal_strength.sharpe_ratio():.2f}")
print(f"Max Drawdown: {pf_signal_strength.max_drawdown() * 100:.1f}%")
print(f"Total Orders: {pf_signal_strength.orders.count()}")

# Buy and hold comparison
bh_return = (df['price'].iloc[-1] / df['price'].iloc[0] - 1) * 100
print(f"\nBuy & Hold: {bh_return:.1f}%")
print(f"Difference: {pf_signal_strength.total_return() * 100 - bh_return:+.1f}%")

## 4. Strategy 2: Hybrid (50% B&H + 50% Active)

Keep half in permanent buy-and-hold, trade the other half.

In [ ]:
def hybrid_sizing(entry_count: pd.Series, exit_count: pd.Series, lth_exit: pd.Series) -> pd.Series:
    """
    50% permanent position + 50% tactical trading.
    
    Base: Always 50% invested
    Active 50%:
    - Entry 4/5: Add 50% (total 100%)
    - LTH Distribution exit: Remove 50% (back to 50%)
    """
    position_size = pd.Series(0.50, index=entry_count.index)  # Base 50%
    
    # Add tactical 50% when entry strong
    position_size[entry_count >= 4] = 1.00
    
    # Remove tactical 50% on LTH Distribution exit
    position_size[lth_exit] = 0.50
    
    return position_size


# Generate hybrid positions
hybrid_positions = hybrid_sizing(entry_count, exit_count, lth_exit)

# Backtest
pf_hybrid = vbt.Portfolio.from_orders(
    close=df['price'],
    size=hybrid_positions,
    size_type='targetpercent',
    fees=FEES,
    slippage=SLIPPAGE,
    init_cash=10000,
    freq='1D'
)

print("\nHybrid Strategy (50% B&H + 50% Active):")
print("="*60)
print(f"Total Return: {pf_hybrid.total_return() * 100:.1f}%")
print(f"Sharpe Ratio: {pf_hybrid.sharpe_ratio():.2f}")
print(f"Max Drawdown: {pf_hybrid.max_drawdown() * 100:.1f}%")
print(f"Total Orders: {pf_hybrid.orders.count()}")
print(f"\nVs Buy & Hold: {pf_hybrid.total_return() * 100 - bh_return:+.1f}%")

## 5. Strategy 3: Gradual DCA Scaling

Scale in/out gradually over time instead of instant changes.

In [ ]:
def gradual_scaling(entry_count: pd.Series, exit_count: pd.Series, scale_days: int = 7) -> pd.Series:
    """
    Gradual DCA-style position sizing.
    
    - When entry conditions met: Add 25% every `scale_days` until 100%
    - When exit conditions met: Remove 25% every `scale_days` until 0%
    """
    position_size = pd.Series(0.0, index=entry_count.index)
    current_pos = 0.0
    days_since_change = 0
    
    for i in range(len(position_size)):
        # Should we be increasing position?
        if entry_count.iloc[i] >= 4 and exit_count.iloc[i] < 4:
            if days_since_change >= scale_days and current_pos < 1.0:
                current_pos = min(current_pos + 0.25, 1.0)
                days_since_change = 0
            else:
                days_since_change += 1
        
        # Should we be decreasing position?
        elif exit_count.iloc[i] >= 4:
            if days_since_change >= scale_days and current_pos > 0.0:
                current_pos = max(current_pos - 0.25, 0.0)
                days_since_change = 0
            else:
                days_since_change += 1
        
        # Neutral - don't change
        else:
            days_since_change += 1
        
        position_size.iloc[i] = current_pos
    
    return position_size


# Generate gradual scaling positions
gradual_positions = gradual_scaling(entry_count, exit_count, scale_days=7)

# Backtest
pf_gradual = vbt.Portfolio.from_orders(
    close=df['price'],
    size=gradual_positions,
    size_type='targetpercent',
    fees=FEES,
    slippage=SLIPPAGE,
    init_cash=10000,
    freq='1D'
)

print("\nGradual DCA Scaling (±25% every 7 days):")
print("="*60)
print(f"Total Return: {pf_gradual.total_return() * 100:.1f}%")
print(f"Sharpe Ratio: {pf_gradual.sharpe_ratio():.2f}")
print(f"Max Drawdown: {pf_gradual.max_drawdown() * 100:.1f}%")
print(f"Total Orders: {pf_gradual.orders.count()}")
print(f"\nVs Buy & Hold: {pf_gradual.total_return() * 100 - bh_return:+.1f}%")

## 6. Strategy 4: Kelly Criterion Sizing

Optimal position size based on win rate and risk/reward.

In [ ]:
# First, run binary strategy to calculate Kelly inputs

# Use historical performance to estimate Kelly %
# From notebook 74: LTH Distribution had 100% win rate

# Binary LTH strategy
entries_binary = entry_count >= 4
exits_binary = lth_exit

pf_binary = vbt.Portfolio.from_signals(
    close=df['price'],
    entries=entries_binary,
    exits=exits_binary,
    fees=FEES,
    slippage=SLIPPAGE,
    init_cash=10000,
    freq='1D'
)

# Get trade statistics
if pf_binary.trades.count() > 0:
    win_rate = pf_binary.trades.win_rate()
    
    # Fix: returns is a property, not a method
    winning_returns = pf_binary.trades.winning.returns
    losing_returns = pf_binary.trades.losing.returns
    
    avg_win = winning_returns.mean() if len(winning_returns) > 0 else 0
    avg_loss = abs(losing_returns.mean()) if len(losing_returns) > 0 else 0.01
    
    # Kelly Criterion: f = (p * b - q) / b
    # where p = win rate, q = 1-p, b = avg_win/avg_loss
    if avg_loss > 0:
        b = avg_win / avg_loss
        kelly_pct = (win_rate * b - (1 - win_rate)) / b
        kelly_pct = max(0, min(kelly_pct, 1))  # Clamp between 0-100%
    else:
        kelly_pct = 0.25  # Conservative default
    
    # Half-Kelly (more conservative)
    half_kelly = kelly_pct * 0.5
    
    print(f"\nKelly Criterion Analysis:")
    print(f"  Win Rate: {win_rate*100:.1f}%")
    print(f"  Avg Win: {avg_win*100:.1f}%")
    print(f"  Avg Loss: {avg_loss*100:.1f}%")
    print(f"  Risk/Reward Ratio: {b:.2f}")
    print(f"  Full Kelly: {kelly_pct*100:.1f}%")
    print(f"  Half Kelly (recommended): {half_kelly*100:.1f}%")
    
    # Apply Kelly sizing
    kelly_positions = pd.Series(0.0, index=df.index)
    kelly_positions[entries_binary] = half_kelly
    kelly_positions[exits_binary] = 0.0
    kelly_positions = kelly_positions.ffill().fillna(0)
    
    # Backtest
    pf_kelly = vbt.Portfolio.from_orders(
        close=df['price'],
        size=kelly_positions,
        size_type='targetpercent',
        fees=FEES,
        slippage=SLIPPAGE,
        init_cash=10000,
        freq='1D'
    )
    
    print(f"\nHalf-Kelly Strategy Results:")
    print("="*60)
    print(f"Total Return: {pf_kelly.total_return() * 100:.1f}%")
    print(f"Sharpe Ratio: {pf_kelly.sharpe_ratio():.2f}")
    print(f"Max Drawdown: {pf_kelly.max_drawdown() * 100:.1f}%")
    print(f"\nVs Buy & Hold: {pf_kelly.total_return() * 100 - bh_return:+.1f}%")
else:
    print("No trades in binary strategy - cannot calculate Kelly")
    pf_kelly = None

## 7. Compare All Position Sizing Strategies

In [ ]:
# Comparison table
print("\n" + "="*100)
print("POSITION SIZING STRATEGY COMPARISON")
print("="*100)
print(f"\n{'Strategy':<30} {'Return':>12} {'Sharpe':>8} {'Max DD':>10} {'Orders':>8}")
print("-"*100)

strategies = [
    ('Binary (100% in/out)', pf_binary),
    ('Signal Strength Sizing', pf_signal_strength),
    ('Hybrid 50/50', pf_hybrid),
    ('Gradual DCA Scaling', pf_gradual),
]

if pf_kelly is not None:
    strategies.append(('Half-Kelly', pf_kelly))

for name, pf in strategies:
    ret = pf.total_return() * 100
    sharpe = pf.sharpe_ratio()
    dd = pf.max_drawdown() * 100
    orders = pf.orders.count()
    print(f"{name:<30} {ret:>11.1f}% {sharpe:>8.2f} {dd:>9.1f}% {orders:>8}")

print(f"{'Buy & Hold (Benchmark)':<30} {bh_return:>11.1f}% {'~1.0':>8} {'?':>10} {'-':>8}")
print("="*100)

## 8. Equity Curves Visualization

In [ ]:
# Plot all equity curves
fig, ax = plt.subplots(figsize=(16, 8))

# Buy & Hold
bh_equity = (df['price'] / df['price'].iloc[0]) * 10000
ax.plot(bh_equity.index, bh_equity.values, label='Buy & Hold', linewidth=2.5, alpha=0.8, color='black', linestyle='--')

# Position sizing strategies
for name, pf in strategies:
    equity = pf.value()
    ax.plot(equity.index, equity.values, label=name, linewidth=2, alpha=0.8)

ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Portfolio Value ($)', fontsize=12)
ax.set_title('Position Sizing Strategies: Equity Curves (2023-2026)', fontsize=14, fontweight='bold')
ax.legend(fontsize=10, loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Position Size Timeline

In [ ]:
# Plot position sizes over time
fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)

# Price
axes[0].plot(df.index, df['price'], color='black', linewidth=2, alpha=0.7)
axes[0].set_ylabel('BTC Price ($)', fontsize=11)
axes[0].set_title('Position Sizing Over Time', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].set_yscale('log')

# Signal strength
axes[1].plot(position_sizes.index, position_sizes.values, label='Signal Strength', linewidth=2)
axes[1].plot(hybrid_positions.index, hybrid_positions.values, label='Hybrid 50/50', linewidth=2, alpha=0.7)
axes[1].set_ylabel('Position Size (%)', fontsize=11)
axes[1].set_ylim(0, 1.1)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Gradual scaling
axes[2].fill_between(gradual_positions.index, gradual_positions.values, alpha=0.3, label='Gradual DCA')
axes[2].plot(gradual_positions.index, gradual_positions.values, linewidth=2)
axes[2].set_ylabel('Position Size (%)', fontsize=11)
axes[2].set_xlabel('Date', fontsize=11)
axes[2].set_ylim(0, 1.1)
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Conclusion: Best Position Sizing Approach

In [ ]:
print("="*80)
print("POSITION SIZING CONCLUSION")
print("="*80)

# Find best by different metrics
best_return = max(strategies, key=lambda x: x[1].total_return())
best_sharpe = max(strategies, key=lambda x: x[1].sharpe_ratio())
best_dd = min(strategies, key=lambda x: x[1].max_drawdown())

print(f"\n1. BEST ABSOLUTE RETURNS:")
print(f"   {best_return[0]}: {best_return[1].total_return() * 100:.1f}%")

print(f"\n2. BEST RISK-ADJUSTED (SHARPE):")
print(f"   {best_sharpe[0]}: Sharpe {best_sharpe[1].sharpe_ratio():.2f}")

print(f"\n3. BEST DRAWDOWN PROTECTION:")
print(f"   {best_dd[0]}: {best_dd[1].max_drawdown() * 100:.1f}% max DD")

print(f"\n4. KEY INSIGHTS:")
print(f"   • Binary (all-in/all-out) vs position sizing comparison")
print(f"   • Hybrid 50/50 provides good balance of upside capture + risk management")
print(f"   • Gradual scaling reduces timing risk but may lag in fast markets")
print(f"   • Kelly criterion provides mathematically optimal sizing (if accurate inputs)")

print(f"\n5. RECOMMENDATION:")
if best_sharpe[1].total_return() > bh_return / 100:
    print(f"   ✅ Use {best_sharpe[0]} - beats buy-and-hold with better risk-adjusted returns")
else:
    print(f"   ⚠️  Even best strategy ({best_sharpe[0]}) underperforms buy-and-hold")
    print(f"   Consider: Hybrid 50/50 to keep partial exposure while managing risk")

print("\n" + "="*80)

## Next Steps:

1. Test best position sizing strategy on full history (use with notebook 75)
2. Combine with other alpha signals (trend following, momentum)
3. Implement in live paper trading system
4. Monitor real-time performance vs backtested expectations